# S02-C3 · Huấn luyện thử YOLOv8n trên một phần nhỏ SKU-110K

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenhuynh87/AIoT-Smart-Shelf-Inventory-Monitoring/blob/main/notebooks/S02_C3_yolov8n_sku110k_colab.ipynb)

**Mục đích:** làm quen trọn vòng **train → val → dự đoán → tính chỉ số** với YOLOv8n trên Google Colab
(GPU T4 miễn phí). Đây là bài *làm quen quy trình*, **không** nhằm đạt độ chính xác cao nên
không đặt ngưỡng mAP nào cả.

| Bước | Nội dung |
|---|---|
| 0 | Cấu hình, kiểm tra GPU, lấy mã dự án (`fusion.metrics` của S02-C1) |
| 1 | Tải SKU-110K bằng loader chính thức `SKU-110K.yaml` của Ultralytics (~13,6 GB) |
| 2 | Cắt tập con cố định theo `seed`: **400 ảnh train / 100 ảnh val** |
| 3 | Huấn luyện YOLOv8n (10 epoch, `imgsz=640`, `batch=16`) |
| 4 | Đánh giá trên 100 ảnh val: mAP (Ultralytics + `map_wrapper`) |
| 5 | Dự đoán & **chỉ số đếm** (Counting MAE / RMSE / Exact Accuracy) bằng `fusion.metrics` |
| 6 | Trực quan hoá, lưu & tải `best.pt` + `results.csv` về máy |

**Cách dùng:** `Runtime → Change runtime type → T4 GPU`, rồi `Runtime → Run all`.
Không cần Google Drive, không cần API key. Mọi kết quả nằm ở `/content` và **mất khi phiên Colab kết thúc**
— hãy chạy cell tải về ở cuối notebook.

**Thời gian tham khảo (ước tính, chưa đo trên Colab):** tải + giải nén + chuyển nhãn SKU-110K khoảng
10-30 phút (phụ thuộc đường truyền S3); huấn luyện 10 epoch trên 400 ảnh chỉ vài phút trên T4.

> ⚠️ Ảnh SKU-110K rất dày (~147 vật thể/ảnh). Mặc định `max_det=300` của YOLO sẽ cắt bớt dự đoán và
> làm lệch số đếm, nên notebook dùng `max_det=1000`.

## 0. Cấu hình, môi trường và mã dự án

In [ ]:
# @title Cấu hình thí nghiệm (chỉ sửa ở đây)
from pathlib import Path

try:
    from IPython.display import display
except ImportError:  # chạy ngoài Jupyter
    display = print

# --- Quy trình train/val (giá trị đã được xác nhận cho S02-C3) ---
SEED = 0           # seed cố định: cắt tập con + huấn luyện đều tái lập được
N_TRAIN = 400      # số ảnh train lấy từ train.txt chính thức
N_VAL = 100        # số ảnh val lấy từ val.txt chính thức
EPOCHS = 10
IMGSZ = 640
BATCH = 16
WORKERS = 2
BASE_WEIGHTS = "yolov8n.pt"   # trọng số COCO pretrained (tự tải về)

# --- Dự đoán / đếm (ảnh dày: nâng max_det) ---
MAX_DET = 1000                # SKU-110K ~147 vật thể/ảnh, mặc định 300 là thiếu
NMS_IOU = 0.6
COUNT_CONF = 0.25             # ngưỡng tin cậy chính để đếm
CONF_SWEEP = (0.1, 0.25, 0.5) # bảng tham khảo: số đếm thay đổi ra sao theo ngưỡng

# --- Kiểm tra dữ liệu gốc (SKU-110K đầy đủ: train 8.219 / val 588 / test 2.936) ---
MIN_FULL_IMAGES = {"train": 8000, "val": 500}
MIN_FREE_GB = 35              # tar.gz + bản giải nén cần nhiều dung lượng đĩa
SKU_TAR_URL = "http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz"

# --- Mã dự án (S02-C1) ---
REPO_URL = "https://github.com/stevenhuynh87/AIoT-Smart-Shelf-Inventory-Monitoring.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/AIoT-Smart-Shelf-Inventory-Monitoring")

# --- Thư mục (đĩa tạm của Colab) ---
DATASETS_DIR = Path("/content/datasets")      # loader Ultralytics tải SKU-110K vào đây
SUBSET_DIR = Path("/content/sku110k_subset")  # tập con 400/100 ảnh
RUNS_DIR = Path("/content/runs")
RUN_NAME = "sku110k_yolov8n"
REQUIRE_GPU = True            # dừng sớm nếu chưa bật GPU T4

print(f"Seed={SEED} | train={N_TRAIN} | val={N_VAL} | epochs={EPOCHS} | imgsz={IMGSZ} | batch={BATCH}")

In [ ]:
# @title Kiểm tra GPU và cài Ultralytics
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    print("Đang cài ultralytics (khoảng 1 phút)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import torch
import ultralytics

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError(
        "Chưa bật GPU. Vào Runtime -> Change runtime type -> chọn T4 GPU, rồi chạy lại từ đầu "
        "(hoặc đặt REQUIRE_GPU = False để chạy CPU - rất chậm)."
    )
else:
    DEVICE = "cpu"
    print("Cảnh báo: đang chạy bằng CPU.")
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

### Lấy mã dự án (S02-C1)

Notebook `git clone` repo công khai để dùng `fusion.metrics` (đã push ở commit `d27b162`) và hàm cắt tập
con `vision.sku110k_subset`. Nếu clone thất bại **hoặc** repo chưa có file tương ứng, notebook tự động dùng
**bản sao nhúng** ở cell ẩn bên dưới (đảm bảo notebook vẫn chạy được khi không có mạng tới GitHub).

In [ ]:
# @title Clone repo GitHub (có dự phòng)
import subprocess

repo_ok = False
repo_commit = "n/a"
try:
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "-q"], check=False, timeout=120)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
            timeout=300,
        )
    repo_commit = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    repo_ok = True
    print(f"Đã lấy repo ở {REPO_DIR} (commit {repo_commit})")
except (subprocess.SubprocessError, OSError) as exc:
    print(f"Cảnh báo: không clone được repo ({exc}). Sẽ dùng bản sao nhúng.")

In [ ]:
# @title Bản sao nhúng dự phòng (không cần đọc)
# Dùng khi `git clone` thất bại hoặc repo chưa có file tương ứng: nén zlib + base64 của
# `fusion/metrics.py` (S02-C1) và `vision/sku110k_subset.py` (hàm cắt tập con S02-C3).
# Được sinh tự động bởi notebooks/build_S02_C3_notebook.py - đừng sửa tay.
EMBEDDED_SOURCES = {
    "fusion_metrics_embedded": (
        "eNrtPWtz20aS3/kr5piqExCTtEhbSYUVpU6ryCnd2rLKUrK1xeJREAlK2JAEFwAtab263379mDcGFO3Y2d26VSUmHjM9PT09"
        "PT3dPY12u906eZ8sNkmV5SvxJq2KbFqKN/lss0jFPC/E0Wl+KS6WSVGJi9t0MRenq/fpqsqLByi1yuAiW92I6GJ/0D3ux71W"
        "67zI32eztBRllaxmSTHL/pbOxFJCTqZFXpaiui3SVKSm5SpLi3LY6vcs+Mf5ZlUhdInWsCWE6FqPj05E9CZNVuLouswXmyoV"
        "J0WRF7FX7t2bCyj4Ls8rQaUv/rpJCrfsyX0yrbiGOJpON0UyfeiIn1eztJjSw3dJlXbE2/f2fWvQE2+v/5JCzR/TCn5sEkbL"
        "o3PxpyJZr9Milpif5j+Lab5cbyrudHT/cP8ggEri/uHuFsm9TKpSonQEbSU3qTgv0mlWUvHjt8dvRX+/313nGeCAFac59nGT"
        "b0oBj9JinS8ItgRyQWOwyFepONsszx8EYPVf+70D8VxdDfd73x2IabKYbqAmDPh/ip8XVZEsHirsRpGWm0VVivQenk3hfetF"
        "T5zgCHVfp+/ThTheJGWZzbOpw0Gywxr3jniXQhuLjnjV715M8wKI+SpZlKk4WiTFksgpoldH7+KOuDx//gr+O3t+eSY7sVmv"
        "8wKQuM6rW3GdrRJgjpKYMVmk+CJKeze9jvgxK5MbYKwloNcRr/O77kWVT3/tCPrpvt1URBZBlFtCv7LuFNGXwFLslgZ2efTH"
        "E0D75OLy7fEfO+L0x9cnAsgDBJ8vMhjyZQ5MHrfaMINa8yJfislkvqk2RTqZiGyJCEMzq5yHumy15DMY4VsuXz2skTvl86PV"
        "A3ZgCoifwjgm1wsg0eushPu3awSRAPEu0r9u0tUU3lxu1ljg5xW80bBXm+Ua+KkUq7V6tMbxL/HZetZqAQ9cp4U45HqjDMk0"
        "X+QJ/KzWvRW9HbeOiiJ5eJ39muqCqt0RYDnmsjCzoVQHwPYu0iJLy3GrNZnAEEP3D8WI6Nyeyik4WSZpu+M9K5alfpji/Jsk"
        "cuappxs9/yYFMIh6nL8PPZXCJJ2oBkyDOOPSSZZvAo8ANeDYe/9NslZPlsl6csfz2DQFjDJZK+aeFMTbk3m/oYD7mEu7z0xV"
        "6HH2HjBTUnDCXAmvgcCtr8Th5/wDeChxz345Obt8++7P4vjtz2eXp2c/iTcnl+9Ojy8+e3ut1iydC6TMGkQwD9WEOKmMqP8P"
        "k6rYpEOhebAjH0OVWe3xrMjXq2QIgiFfANddQtVOKxbdH3h+jGxGNdfjIZO63f4lWWQzFD1SlAJjVeKmAKxmXcADpA2+wKZh"
        "YqZYBGV/KWcDidxc9H/kOSS4Hz2UCAgfOzJZwAwGzPAn4r7FIpsLkAsCWGOFa+Q0lW9sHGORonTkF70qJwAsvRAdDy4+CsPF"
        "N0G4+MLAJcBQf5GuIo13LP7jkJ7oFnkho78iyQAOEHCT0kIa6Tf4N2+/Tlc3QL9lVsIEm4LYTqu7NF3JHonog9vUY0ykZrzk"
        "W9PsY9xra/iN2B4eiv0tCLZPVzC15SgpPKxGpySuAVGRLtfVQ68tG3qPTCLZEgXyiEZ7jFJubBVgBq0VoBKoSIGMXQPDiL9l"
        "a4N0R4RoW02ycrJKAACOpjjD5RsgRNbIKsHNZMNFpZeVqwToEcdYFsQy3kdqxUO+0VDXTVDXTVDXLtS1JI0cB4UvluDLocMN"
        "UETOVOcx/kn9JXVeNA0dFk4AWcL9+VlyhqTfwDy8u81AXeU2Dkmv0KPnjmAP5fhqFlEvkVZeGZoVThnoucNvBtbTDHeWM1gR"
        "rfJVF/CNYT3OCtSqltAPXP2TOSz2Yp4t4AfuNdpFCqrECiduUhLHWg13xAx0h/QQXhKO37yMO/WSPPNrJaUItlfmzyt5qSEt"
        "Yen3WGqXqQho66DCH53ENEWyKl1qzAxhUc8/FFEflFag4Nei3CyjvzNekwz0Q8Z8kv1d0u48KRLYb8CGgm67+s/qp7B6pBv6"
        "yZb8BplS4AblNmU1sWeRJQjlXC8XNoiIdL5fpDb8pzS7uYU5DJ2+SFcl/Lza4Ju4Z1FXMHlhENN5AtrqIVJZN3M6Z6pLlpJz"
        "A3lKzWycILjRyW5WOfI1U+cdcZZDGpZROG6G5oGBAuFFPdrAtg869ANwf29fYqyGGjhQ8iheEY0OG9Z7teipRUpOX/6J7VnA"
        "MxGYeAlY4W9yXUYaflc3Gsc19kYl83fk7/AmEzafsAXdyuO0Rz0U5V+LKnI5PQpwevw/Aym6fh+6E1oW/UvqXMMQqDFwdfpP"
        "GYSknIDWPQX9FzbCeixIvm8ZJVpBc9gY4pI2lLrZoein3W92GULbFPCGVBdlEDCjpZ7UxNJpWDCJ7w8NSvGXElOfJpl6dVLX"
        "BA+R3JY8PAaST2A9w00uyocRSIQO7Cn2xz0joFQpqwFTdH8fC39JqefwgxiqDa8Cj3yhwcMGGdZh2J9ZNXDWgsYPi2r1IKa3"
        "6fTXj5KkzE+8c1CT4UvKTFK3QSc6FFsEpcOQjMR0ClVcSStBxY5QwIJf87ihTuQyDm0uoIQUAd4G/l9GBuBiP0d7F5q08rko"
        "kyVsJ1HPTIGz/P1ghrZO4Jpq8QAqIlk3YQHgbaSgWUrgpSAX36u53bVkQofMQWgYAB5LNiWUu34Q77MS+E7k0+mC1APkb9hR"
        "4duZsjiWX3QBRmr4bNHAUaJriziLYwjIdpbBIpJnXOvO/wOWAYtlgpuA7Vzzg+KaZ09yzZwMq+u8zCowJVmcguwDhty0AGMi"
        "ijUwYfyTMM8P4rPwTs0G+M/EPmjcHcGoS6vtSK5DYEMajz2uYiMkWLcXQhsCtdKofSmwiCYCNu83sPddgxn7iWVpW/t6tcJC"
        "aGyG9u4yUC9+TR/KodiDXeJeR+yhNo2/rmqHTzxJv9fREPfcCY2FK7CKLyZyhhh4crlxwdGtBlHufUmGnWXzOS6cARbl1q7L"
        "iSojF1e6lQoddQteAUUjNBUYnZheq06Wv2ITGlRtJaa+q3KyjCVbqQwSxC1izaCW1dx0VUmMUG0HDdVgIdHi5mrlDBayHDVZ"
        "K6YRUVptCdIIZ/22GQv6oWPl+KCZpY2OgqEnMjSt4tiwVZvcB8Mn9yiyolPV8zYoIIZiz3ksSavH/lh1fZ+EqmzIuKWy57lQ"
        "dTVpt1R15gzUpPtan+QUgve6M0HcsYTGOIQgFlBo8fvHL+J+QBfqH/775PhS/HhyCT+nb8+U80FE4C/tiKPzDvgkyZv67uj8"
        "/ORd/KW8ElU+QYdsdJ3fpyD0bOcBPJqwe3aICzfwdxuLtkmwm4JajB9LX8I1ruYoswkm6QW3yRrM370euBhfxjCMYnTfB6kE"
        "/98P4Hcwln7gniXjpLyR9j2CFTLt0eZjiTPU4NtDPWMdxb1FfpcW0oUA05LKqW5YRkyp5BdFveDdrVUw31SMFRnLwYL9K9kD"
        "mtDCv9tkMZ/csXwdEQGgs8+BBfbdIrd2kRd+EWiY3+yP7XJw15Ut1Iv2naJ9XfS2XnTgQ33WCPWFD/WZD1VSE2q0ghbiefvn"
        "VcnebVTd9KCJvQ/m5nEPPO7365TUxz0crz1U5PZwQPbIYCxNXtqnyVoPQOgPtct4xL7fcUe9G2x5F+T1LVqw1FdOMQKhlIEQ"
        "KD1Y06CJHGsPUHWXi8GP3tTo7WITwR7RXByIOvKa6C+76YJCAJwmgD45xKKsQO5KpdciN3XUmASYyBrgsa4pFF3kMET+3I3N"
        "yODLCS57aYFKh7q664jbcfxRFgQMGyFHh2dhccbguo+ajxJhrrToBz0Gpv+H5pJn6/WgGdjgY4GBDLnu91jugUMxetkhKl0P"
        "3GfbHCknCXoQ7dFcbkrtD+K1DrZAL51BVq4UiouZ3CN9lsl9dN0f7UMMw/UAfmKrwINVoM8F+k6BeyTKMlthgQEXGLgQrAIv"
        "uMCLsYPEnWyCxlBD7WoUbWi39bIPpuyDUxYUnYT1MmrkawWiJReQNOnb0KgDAIpIgcqG/eaFfKM6j7UHTu2BrD2o1R7I2oZ0"
        "GxQACj9G5JkE2bWQ1y42q/z35F+oLU37ci1wbNQWGZ5bMAKyUcZ7aBGZliAkvSgXX7Q4IS9GSqbl4FOrbhOwAXXCkbJo+bzD"
        "GYJygXtjKRZnHfHGlbVlWpHm4Qtcbx/VDygZ/ablHMoMAuUHQYejHFpso1dCNCD5TVEEIBTzxIz0SmCBehW2LtIz7Cywn66y"
        "tKoMAlUG9SrGxfq3FIISowj8csu4sQeugCUsIDgOgUbdPupyHyFRCaEda/syDLc4yX22hM0XzNJhhwzfqKnQ/OObId7XhFtD"
        "zb5XMyD1sCZY2f2aA69mQBw21Hzh1QzISQvbXcVluMpvlJo+UO6Gkp9MaagbLvbCFPPlaa3CwMAdbIE7MHAHNty6pFXkHiuR"
        "q+g9DsheMxvIYhlZ4H4QFjVrMrZDXmBP0CbrSEJF42RZs7Hp0Lj6K1D/bvOZlozTfJpDyOsO2mcgYPboHJyu6HdnPFQ0l3wN"
        "2//3O2qesh9BrxqEQecVhEKzGRYjPWbpFAiDljnVsIxTIf0Nh6znUSEI+DgvQEasc5baBu8wMKZbTZWV9DPa7KkdJqyord92"
        "ha4xtEKNndhioQzNby66FJKMZLWDhv/89vXbngtSRSkDVHaIER+RCYJM57p3XUmxFDbQixxXtF/eHg/2+/vPvEgDKOeuQHKM"
        "AgK8VyQAzITPeRXNMGyrq5YxKOyuYmvnSVBfoWdfiSMKLIIFGamRaq6ADQg8hpkQKd2e46+iPt7S7KIB1ogDOSHcOkU1N4pw"
        "QzBGJ+u0I0a4L5D2uiXgFSrep+JrLr/P5SV+aiZtMNjVYjgzFkuH1y0+p1m2SOcVWjQKDG7xsYCo6XVkpFkPzW9L8p1E6i0W"
        "jhU6GF+NaicxaJMVgwtZYsImPwXl4kmD8jZfzKTRdgHxY+sEYt0UrdEH3TdGiq/EK/SS4IZDVxUwoeYZjMgcfM44GWbpvXTs"
        "0KBAJExh4uFguk6V/7VMk2J6W9L+PloSzWt4dcCGP0sP20i9tkEEwo6BPyQc0lDYyFKr78Ww0YrGKHxPEWtEVR/siMqiAQNf"
        "j2QN+bSmI3n2WAahzNoLdxzUTLeGIpvdczd4aUE6jPqwCsHOj66HXVjEAqqZse1iHAwVRVDP2IKj7nkPQr2QLzVmZbplV+ka"
        "X8C+mAUkI1hi+OLRMrVMVqibLWDKT7IlLDkT42LjZQ/DK4YU0s+35UQ6AKFIKKaIokaNfwaj7D2X0JlqEaMoutSqcewhT8G+"
        "V8Yjo1jNUHuk6GDU/LHl0lBi9KF9DRomWHh980WHBnDe1kES7eminGQzeACgHzsCLF1jBy8Ye4olkQGlNemnImBBuCRo+ecI"
        "WY2UhQsZ0VTz8kY1Drd249ioCVrF9jsi4nDaCuO+Y8uYIMM2sZAfsOmiaVXwgJO9ANGN3bqaC2YmFlgHQIMQmamIuXrYK/YZ"
        "XTe9m7SKmAIdfQc3VmCqhRfVqlF6e+SsOZk0o1Mrtt1kDwGyMZEuwNnXa9db5mGQfh+JpBwajfQiuU4X5hZew80+RYMFQn7n"
        "2i2syuOw69olHhBq80oYxzwi1gwyTiQftBkRFcSr2YsC3fE6NlyGPzab8cVjHOIQA9phaLAA4LxFhpaB5R/atBXWTEydMbdE"
        "KHn72MjNHrex9+CQmEkzSYoktp6oR6OxRXRuzqkqMbDrcsepMli8v6YZQ+Ds8fOHweVA7qbTkOy50xBQnJpBLaSpIVduO8Dr"
        "9Vo1iePU9MY/9iRcVMaekIum8WNtJl/DGt0RUxW+Lx0wjFVH0tjg74q8F0N59A5s3sQiYaeOHPegtYXNGlt0zNHYLgdr+RLX"
        "1sE2k6qRC9Q+CwY4+zD4URtOQ/INqVHkdxR7UBTDHZlDCmGoGKOy9M1wh5kblG96OAHUaPhyrE+w+OOK71+Oa4OLjw/G3gjz"
        "oRL7jnQajTGQ/OAfgvH+03iWgYWgPtjnRnbyaENjJQ/5LWxvxAF7MkATAEXgYdBBxNiX8Q0IycVmuSrthaHerDfAX5RcpLPX"
        "h/XlJwzry98Rz882mE74b3A4X+LIHfgjF1jIWJuVZ3snuIebbMw+PrKuJ/n1X0ij9UKZUFXVmuoJA/JOfoNmrSKWaJcIUW62"
        "sQAEkTozDVi/5xN5rNTi0WOR0wlr7Q3/Sh0Hli84TKmHmlFSAZhr3MBGCgOxCbakxeotiF+o5fcUxo20MGcFnqheHAqvOLZu"
        "LNHJ+mBf6zew6lEDVv0OnSk92G/z3j62a3578HTNbw9CNQ/2J9/tULledf1UpUCd4qk6Ra0OYcirXriOJMlo7OxVt9fwyq8x"
        "XgUPdE/kGHxA4ZDFSsS+l0cwOuI9LmDpCqO90QChkIsfaQ1VqIKqjScCKbKcdM4Pjw1tMe13b85u7MmmalFS6lwyEGzIY9+p"
        "vQMuGTJHdQL1AF9VFS69EubcMhRZey/l6WV4U/jVHIpACffB1tKMT+2ZV6fMNxBIBgXb1gRsm0KPLU9E0JYHNRbOYjChexJC"
        "lgSSwlG+BWe0hudPcxuMrfk0SxG7hreACr+4txGoAaM9AVdFNmFocWALofpS20a4osnb5XAdVtPlHHsOsvtgP/pD3O74BeRs"
        "9Z5icTXz41Z9mavJqWYcaguhi1T3u4NGvJCZgqh1v6uLThfBoFwM00bPkhAe1tF/nxqW8AwD5hkWgqoyB/ggP11ABPGwl5h/"
        "mPD48LiDwKgVCkoInn01McGa1SVsuLwD7I5d0IajkqFIlcREZTXoMxEqBrFoW/GV4PQWMxNjbZ1f2vMn3R642faIynvy9Luy"
        "P1opKSKdEoCBghdBpQtxs3agqZfcf1SBjxlMSIfcsUqAEFZFLO8Ux1ALY6MONcGn5N1azaER2m2rDLRNHsIG3ZR+IVhjnkmd"
        "VObmEXKoOTACVEl7LC1tlMe8JMdMqbPqtOShMdjeq/CKLqqiJrPRTu5Fa/zsyDbOtpJLypljfNoCbI6VGAs0HSr2dOwmrRoW"
        "Dg31yuGJK9RI8mVWVeoInfP6o7D0EmronDh8VA0Jjx7ABSeMAEPalUWOq14T8wk2sNcbtjt+hY6JRQ/6H8VXXt+jKzNVr2iz"
        "e2Uv17Jll4/tfjP/BhDAuByrCpjk1bmUHkgH8svS0yvXHXVAzj4QqOCQiq8+MkbxD+FQRxmwKCI/blQ6U12Xx5O+Y5g25DQP"
        "u0oiXYHasZy+8c5HU5CZQodRjKB0fNUsHtVxG8q5Bb8wAIdIz9gr+u1BQ9FvD+IaVFhhrNJ0HP5cpS9TDUDyrH3OoOXUd5eq"
        "PTYRW9mWxs2FudVt5V2G3HNSjtjlYLfBcFMsBMB8v01oTlnbkJqRcTc7gYRk/OW2bLN3OShIXZmikpKQ79sKCWfl2oKjtufI"
        "ZuHXAb07/ogOWcAaO6LOGwf7sGuenLbMlidSsGLAYnQVIOWVePuOlyZHMBKZPKHdmCXH7pDK6uNU/QyZfWz6mAQ+qlWZ4Mel"
        "FhdzEWnI9ePJ4RqdHT//iCDqLC8QVsVbcfTdiy1SdxzwG28H/NIC7KI49l2Qlsc+5Iis2fm8HjelJ0J7HhneCb9trmnI7uPY"
        "6OlkPeNPmYms4Rob0DfVk4BvfMB0WJIh3yBkZ4SNeyRfLHAhxnAfKakQBwyPMZWEYhUrExdPQcx2JX66hOhvEIDSpHdTKZmH"
        "5pjZSJlfx2yJWd5Qd7KV6Zpx1PLLR5Nj6wlITHMFi+8caPSI4aECYMGTASEGW3JyWsjTLl+WsnEx8wF5wYY63HE3iGEv4b1g"
        "8I3aadVffqYdm8vleLDNBMa4Ja0FDRFq3PkZ7XxCCRFDBiKZ8xSCBs+jKcaMQasyPHMtI8StwJuIjnPa1IaZ703p5sBh5Ikp"
        "sM59R3nSHWOgA3ZoxSH9lNCSgLMDuJxdwLhhYaWapM4tiEGqqqsBD10/wAy9sZUIKwIdmdlYFvlQ4Q1avYwzXfE3YWxNGYOy"
        "mjxePAT2TnmqRzBjyEMyrs0wSgdmzSeQhXw99l1KGmDdLaL7OZKojl13qq667UBajQzPeGXUlXdx0DRhIjlnn4O7G3ijNtK2"
        "hHOHWOmedCgcWcAMOuJLwkYqg5z80M7uaemI47EbrVIbbi3XvAGn596Q1+XdMBS4EhzucASL7otyv0UaORkxMmIHG7pKNZPF"
        "Dj0vKPOpRclr2kDMQdHChC6QtxVOY+FuqU7BHgrdCAJhDhfJ8nqWiPuhuKfzQAUk4oTDbbxouunv9CySEaGKh5higcVeC5mR"
        "FAxDHJR9L6JF58ZzxqriGiB63DGyZJHbWLW2GdLF7anZMf8Ndfks8gwIs0UW1bmFR3votsqzUbWIAXT1WewwMssh1CLU9Oxh"
        "hEUZufUeWzV2XjMAw3YTDhNaT8iY59DcECMsF6QwrIuI0CQhVVhWCjGNGZPRWsmZUBBUmHV0Q5gq+jB0ImpEPQROV0g0HUmx"
        "IzWtsFkMkL1OwQe9Wcmhl+sWubnIVFNo17CgwHFM1jx7YE4R0S2EDWN9tJ/g6VHKFw11IaAAuR0VQA26PvzYNPDahINNZYqC"
        "pLjB02nY6bihCrzTBnksN7IAjeNWMA5P1bMwQ33McLweaAdaeDyrHcbzKcgynLRWN7xkSfUbU1nx2OS0/jjjBho82K3upyku"
        "BiYSmrztK9g4C2u0gg1oaBMnIppDgInSknwxHO+P/jfQwzjIadZsqTWBiWQaZo0eb9PLwzqOozrfjGqFALFxYxu7jObWEdXN"
        "NQ7q9oENiQhvPZhA5L08GrBZYmx1tfbFfq3IfO2HRfKJmEMF77mcQ2o1dItbR12sGtbhJvnsmWy7g5lz+gNv/tXWTc391sEj"
        "fQ7EPtfBJspDL0e+3A2czvBExvxBRvbjKoCGAZrXaByU1h/cmLZkPPuEHKkWt2Q6MaWlhdpb/i7BVG47BEHBHh8LAkyVStkJ"
        "hztMdfyBoRaEIDDOYx2SMN2yC3lsbY1wMC1YJwJMS0OYH7u10pgDhjeUTW0MTW/srC6uT7Ox2rcHtWpqc9pQ0ym+c5TDx0Y4"
        "7LYT9nbB9X1pa/cd8ZfJ6ILfh8B88t3X8PNanL87OT69wLQuz+FjCsdHr1/Dxav+l84wXyZzMFFl7yPosJXycaWvUbfEAiQg"
        "TL4udHrXzxleADQ875SRrRYPVfO3EkqV3BI9L1cOyCs8ILTCJsGLtqIva8BObr9njq/ZR1xIGkJZsgPBL5+5ZDMQF3GA64Mo"
        "SuzxRzE44ZWTSZ9Si8ttITnqrMxn4Xcqb9yEwprlKRYrpz5p8c4/YyvtDR8giy4hac8r/B+Oo1+esThIRAVSLq28Fky+fIq4"
        "oH/wAHi10lYJP3261bWO3RcnhTq0QgnUQZF2G7QSonOZ9ZYylN0ci5H5Ea9qmyswGvRbTogp6oJba81DtUwNrB+qtfJr+TpA"
        "ZZWQDFZhTnf8H063VyudJ6/hAxaflC+Pz91qfzzzYruRm1An4ZccOL9b+EHzfN3B76/48pO+R4MjAvmc5pyBkxIDl5otS/oy"
        "Uuj7MfbHaOR3aqxv0jzXn6TR73I+Q4jfnXkuPzvznL46Ax6hFVIR3Si9z5k7mBBWR0Qk2nSkmP6DgxbQwdEe4oBJ+BAx/JW4"
        "7Y3jj0g47DTV5PdnvKXrW7JVzSnN7GX5pKkc5UUEX7/jHZZFhzDY/O0dFYaMvn6XM696rgMYbG17mNzjjhK10wesEsrricfv"
        "2B4XYCZklVd9F1TGoH5a5NeQHVB9aAlZCaxSoFkviRCUOtBITdvYJ5d2B6pCCwDLWMfup2PqUkLFVej8yLrdS1d8S7MkLXNX"
        "crT08FwpfgJG4oTNez+eXhz99O7k5A0oB5qfJA9Jxng6qgRCruDrS1mlzzYyErJ+hUmoIVcuLNIOWhTNIke1o8aE7BGGkr26"
        "mKkni7ZNdOSro+POLGmpxcRZ8WEdfPbqHNvBC1gHAQ1s4TeHQ9A2fE8LcMrNSSOLV/M+/mtn55QxTns07mb9YJ+z/KCW+k7B"
        "XrUmIPwvga7oX0orC194AvnImTxjJbIozadWbffENZwK/3WW363cE/v//iDOl/ogzm/5Hg5LZvNNo90/jCNP6Ms5Fjyi72St"
        "FIcehjqvKCofdgJyNDj0d/tqDmlopMDFKn2mas+Moj6eLnUT2812tKnybjJL1hUdzEbhiQKLp7MrGg/7OIEp+wVNQDrInSEV"
        "jS9mjef0r/F8d4NG6SpkmBUorMSHettR4K0zD5wVwmx4ED5qpdSII84OvT2EZcppgLDaEcK87wCAdI1wRpMw+5oTU9D1M77e"
        "DWJSOCCJYohT9RROT/jB7fhkSoLREKFcfwXfaIP1oO89tbLIWpzsV/VEJwJK/BDoao17fj9keo5P57WniD4MjwcBn1b+Uyn8"
        "CTgPqlcrlFlW3fu95bmORgVHzTfkdZgeacwMW/PBq1lJAhpS9bRpdW7j4aFMXqiluf0RUtYO3dY6yQd55WbSlJpK50nFIBhU"
        "xQpHMF6ODTz65DWtKBRej3FmmFA7kieGQ2kqvkJtrUwLmbi9wOxZJYSDAU6YKwfPmYNfNOFPU0gtUYrs6paWE5QWGlyJS0wQ"
        "B7MaP7PEqZmBVfHgu/edbsnoFGzAOtgCDgMQpDqofjsEqOpZMqWnUH8MsylfuRXHwOxKxgO+nFuXbEPQuzUUzMBsNh7DzySY"
        "/y2Ud4EgJyf5AKgvrfqBuhGQsu4b/uyC+59EJssrWz66YSlsYarW3mM2Ic1rj8kCpOhqqz4sVg27JzcUseFxmWyyYzWuG9xx"
        "jBHwx8BdfQTcIE/rnnyt2u6YZ8/MsydbcVMZ8YJkZaUE9lQ6rC1B4hBJjR5r+Ho6HllczIFRU18eoRZLLfH2hdqkHYq99TQk"
        "bmpHToff2gjRu6kNmFefCp/WPFzs1CrrsTHMClwgoEl0OLnNqrkjfVn1xuuxOQSuHoVjhssQ1Ot1Ay/WTa+1sa85Z8mLyt+U"
        "bGKJmJNQfnKH1d9z0+9Wc8IBn4t2xlgx1++PrjUyO2OLbPrlMW3wZdqrlmIR+yMWatVSQqrlrlrcZ+vpU9sNZ2X5pNnz1Dcn"
        "6jsDa1eAwjPkArVdn8b5GHBF/As7IFyXIX0BYpVRwOF8s1J52pSzMF+BPxHN3yZbKaUTYqeC5U4IOwyNINruyVF/0p7mPSNb"
        "mDvH5JZJ/rovPWOIe+sWZcIe8o/7assqbArGjlh0Dr/aH1H/f8ctbL7/N6u4rKLWowCf/Eu7ND+NR5RH89/8ofgDNQCHN8CS"
        "gq3pb6tNmEDcUd7mY4/e1xhDvoM1uKh/MjhdVImJ4KEzlzicdo5P4Keaa5q/FMTfM8U8/sQwVQb+QOf70WeTD1W3D2kwzyZo"
        "9cZMnzy4xj0sx1kppl3pYxvSdwShGjyRUOhzjR7Cppby8A5rtX4QzxprkZ+Y2vq7U4u+fhuqZX/EDanNH3FD2jZ+xM0am449"
        "GB3n4++xGQ/1OTeC2tVNtTTDqzMTODJumjs618BAnDx3swDt3L2AnG7yAEMbB8FJ/kJAfhAfA0MOydYkbF4VHA839RcXaLX+"
        "D5YzfYc="
    ),
    "sku110k_subset_embedded": (
        "eNq1W21v2zi2/u5fwesBbqWOrCSdzmDhmcwim6Zzg0nTokln7yLXkGWbjjWRJV1JbuIN/N/3OYekSMl20i62AdrYfDkkD8/r"
        "c5h+v9/7H5kWsqzEPC9FvZDi6vdPg6Ojw9/FP95fvP/8l0zUZRKng7qMkyzJbkWW13KS53fCuzp8NTj9wQ97vbPPslzXC+pe"
        "yFKKpBIzmSYTWca1TNdiXkop8rn47cMncSA+pSCWrutkSsMKmc1kNk1kJbw8w2DaRFXH2SwuZz0QKeNyLfBVXK6WH9a+qHIM"
        "iWsel+a3yVRM40xMpFhlST2oZVXLGTqmcQpi90m9EPP4TopZXMdMBvOyXilXFYatsukizm7xKcmqZCaZ6GmexhN7zvHYfKwO"
        "cOTo9Idonac5WBNVdyuw6i6a0owwKdbZZDwGP95grUrWIo3X+aoWcVWtlljDK8p8tpri00QdM5/Pkym422LJeGyuIFzHy3Q8"
        "xmHimSz94bDXE/j5pczz+tcD/kw/fDVh/VAL8TlO1Qdig/pEP9+JPMOtLONbKYoYHMGNizTJZCBkeBuK8ID7qgMmFR2Gfxa3"
        "DfntLhGGYdONk8vUdtOa6BbOz3csSiRgy7ge4nzTtBIP0VRmNbaxNh/uxQJn9TIalia4Hb/XeweiAt2QgCJPsroCCwZiOMe9"
        "Dcc4bIJblc01EF/GQgxEFUMU1oPpQk7vmNGz/D5jLs5YDnA3oaVTyVRO66haTdA+tvseQDqxsSXkvsLNBKKScjaYxCQ400We"
        "TFmmLzWDHIKTVZLOtumB4DQv1M1jfiUzw1vFQ8hgnYtYcavRNyVDDvX7MsGZ6RgRy4elzj1MfjymfiM/LOoQufF4mUMvQ6ZN"
        "gtrQJIGO6jx6WD+sx4FppeNP6yTPKuqLyzJ2OudJCuZEE9xfns3HAQRC90zzVVZHdrKdwz1VBEmI6kUpq0WezrrdETMjmuQP"
        "shr3FNMy2JdaSxoMSFHKWaJoK57dkz0Yj+erCm3hUsJkTSscXD4U2AOupg9D15uX+VJE0XxVr0oZRWB+kYNsnEG/Y6bW6+m2"
        "EoYiX5pv1WJVJ6maT+oDo2Qmf8BX1VGvC7ou3X6SrQPxBpsMxDnYFE9S6NoF5CgQ7wtaK04DcSX/fwXTh57rVUEDPmXoaTaR"
        "rZaQlrgSWdHr0UIXCezYsRp1U9VlwMuPYG/O3p58uriOrs7O3mDAYe/83clvZ9HVp7dvz//37ApNXp8Utx8I+i3VhyJTvyfL"
        "ou/3rj5cnF9HF+dX19Hb8wue9NhnSekPRb+xMTQDikdt2thQC5kbHqbNTn/T60URLDDYfCxu2Fj03V32A9W2U4dNZynjWcQq"
        "EsEgNM1KQOgiWJJ4QGsKS7OSFtO+JVqmo6X8ptFVYNPWUbvmALJM5uvOWFedTNsOZTJdbVVqb9lOa7V3lQidkIPed+L4P/kD"
        "ekeheHNyfXJ1di3enF+dvv/j7OM/oIJ/nFyco/38/eV/fM1ebybnonP3Hv0XgVFyKIwqBIKExX73xeBX1rEbVouhYhc0n35/"
        "BD2oesvTVkWawKxghvDG40bEYTgOYE20eI/HvjbMkypPV7XrR2FZmPgFXGklYop8yJXBPlEMUsoUZuUzLHIOcuyc4N/Y4cL4"
        "7nC5WOpnMUnj7I6dM1Nk+tVdUhRyhquAnUim0Be1WimX+WesdL8AY8SdlGyCOLIok9sERgYfEDqELVbQTlh9oJrEKY8afO5i"
        "LrtdDdtVfzKnuMgOC5OKez1/2AQEOE8lxVu0Xub1Wwjr7Kws89Kb968sv4nKnPqG4rGhtoEZUueV8I3H+FV7el3m1dC5XTIr"
        "I+6jwLWM78F6Z18sPrV8qD0Y2HwGthz3V/V88Je+H/K1M4PdbVMDiIJSCNOaFHrl1qkzacfTD9S1TrKVbMdKHQZmcouSHUYM"
        "NILl7maLmL20A6fDJewM5zUyZmObJLWE8Wzm2cH+9ppVGBcUlm+NglyvShO1aEXdZY2diR31pC9dxWSaJLQc+DA9FaUiuor1"
        "aohhjGi7GsyhQUbmkWlBixF7Gs36JYuX8lelWLS26tTxqu5UCt7Wj+1L7PKhiMu6QidbJucyub3RlL7aR99cB/d29eSPOF1J"
        "oyDnNkJfkN/PxQtF44WYJSX8QI5AeJojPMhwaqiOXZt0RwXbH2WRxlMVB16cXF1bGs3MADtHKLGynAw5XqQwpwqT5e0re6uV"
        "Yk4ye6ADy8xTh0RcdoR//OVmOBwcjcIEidyDZ07tcOoGk0lh+9onu7LE/H2paIaUrcGVIisCHY4jfC1lO+MEr23/A4FQPTKm"
        "woRZNxSFqWgJxnlE+7gEE1ga211tuTzVyQMiy9gmxTp7YHVEXoackLmawuBzcqlPNR4/spUZCrOhDSeGLPB07xV/HPAPf9yy"
        "l42cnM/5KmnFAIS1cI/H9EULM+RbZ+9PuDGWVuU3lklF6qRu1kqgu2SsvSPJ4Vzec77ImbVanxhDx7Qcv+HxI7uVeVJWdWuW"
        "OGhnjYJch9oRqUieQcorAAsUSiN9F9V6WdQIrbUdINdargoiZ/I5/4s9m7a8zQgyvNCpL3Rc5vr52me5rJiYfMDhoIUNUaOE"
        "5JIQEZLSN/oQNOLvLKl35bnWHRP9Hbt7eofv1I06ZuLFIwhtDl7QJpwdCi/JGok1bPyrHxrXWyInAY1ZK/w/aoL+o43hpqNr"
        "loN6crgqSFk9O0ZTV7HrsKN4tNbGMo7kiHU5Wa6WtP2GLCLwZcdp6/Ck4yG7yYwWzg6mQZP2hZmBpWc9pNq+JqbNoTmfc6VO"
        "s/jFHGTXTbaMv4qOXijDsXmh1fXRIbbR+w50YotbgnVKZQw9e9TLbMK+3QvwpgmxRmvo4agreDxgVxD3pIPSOj0BtKUtidFd"
        "KAOTbIsZwrAy5iRGCVo3rtoVQTAZ/yu2dsEWxeyHBKnZijI2CkYghABbnakIlKCdhd2Utt7qmr9JOvUqhE/+29mF+Hh28ub8"
        "8jfI6un7T5fX+PgtEyknI/YsuzuxWVaEBLoiMd2VOrnRGWucSorICXiXgfjRh+Wfw5jAXRIJNN8A5wvEA4CzNf7dB2Ix6gB8"
        "2iOeWDEqhYTNX6sV1olEaktrqEasdKhX4jVC8S5OCVYkwJcTJk+5KnZOc6Re2uikq2UGXVRSMx5buQElwMnT3PgVDTEZj1Rh"
        "F1mdrpVEwAC1+NIK8xumttzN3hxJCRo4/k9Z5pWnzhWIGVyfPEYzc/Kn18Yo5/cm+eH/uHfUyYGIA1GWByYZksCPGID3vioj"
        "Cgh8L+vjI2e7Jt7l5IjGbidHneB2d3Y0dwPIX8SPzyr1I/us4aM+22ZoLd+vx+JHc7WBuMUWHi3xlhUk7pmMRrHO++wzyz43"
        "YfnN8MfRqBMp3Ff/zoW1R8cVS6pH1HZeL+voFjq1X0mhdY12oiCCEguFR7clBQMo0qwgkkxBeMftVvKdaiHCM7gg0NHosAmj"
        "dJqXIcLebz0gNou4kPAq/jcxlT8A7ji7Pvv47vwSfvz8VFx9+hvBUFdnF2en3xB5aiGCHmpLM848oIAGyb0xNzIKRDYkPqna"
        "AH+EmrhA55Og1Okiz9kiAY3nC0KsTVgySiTNsugppSocJZN0rU3mNdUQmjFssGZyMDPw0IyTkQooMj5yIB6ouhlhRhXyPl14"
        "qwSX3HKVf7/EgV8yeYiUpR7oDRLJ8ZgOSvm0CdkV2gSQiaQKQNR0oaN/Cu3vqRqowoWw2bcSL7bZCFxQK1T7fDI92p2kaMYh"
        "Gpe3CmojOxiXt8YH0E6yRkvskdqGHIVDAPEEOPFGvEc26VNlI6Z0LjtzY40ELNjhk+l8JpYrnBAhGIzVoTZSWWObmMavbBLV"
        "DvwnqZ1yoUKLJ9FxJYav8dEhtXFPa8IbeNhjXdsIP/Ivj66zZbQ0DzA2rGIK4DRFiLpv7FW0qkgPdGzvmZTb1DOUqO+T/N8B"
        "UzpZpE6jmuyPpGwR4ypjSFg2UJ5/h5FSW2i7QAe5SQxMNdyNkYUKZgjTHDLq+Qaj6RROQHMbr9sVkO7ydU9Z9Z1Yme9DHA7b"
        "hNUxnwHk1CB9OW7xwmvSYgcmURq+qinD7DZnEed8xpK9Pjw07cj/TOuRad1n9PQSCLe5ajKES8pTjLkuIUe9DvRiRaRrHalY"
        "GuPWHwhMZ3EV6likzE1OznHoeKwPRPYga94PGBdHJVRTPndOb5DCRz50gCNuHNRwa7RGW3aMBsBi7bLeIxk2VqCZUlG2sKbi"
        "//JJpIZjLLhwMtk0iJUC3YrpNaJrqMqfVBD3NTBq7HIo3pMl2FYvq1YmY6QIgE9EO4R8NtUGvNy4TUjoEPkPylXGZWh+SIEX"
        "EhBzgmZgYxFdIktqnBMvJ50hig3GqrOgumZd2fWdYtAwHjBag0HcALyt8lU5bRdgwnDUgBM0ZLShHEMZMfJ5PIFeg3AN7cvx"
        "Irpzt0fLgOqskn8yemB3p/Wm2Qrry6Zn7LyhFvJltCAMHW5aZdmD+JzxTOMM3q/qAum3hXvilEK1tbpt2OJHs+TGiYRVITss"
        "lzUe5HhmhMFm+EnCcPeV7IRoMlY14sU2MrMXYfkCmGYn9tLY+53OZwt/UYMoy8ieyTJavXzCLSiGQFDHwyrSG7MlfVSPYCr1"
        "+gjO2Q9Ff4ty/1wpiUGRDf7WYMh/bc+xp/pOfKqkAWUHFXKgBIYEtgDsXXIOS++Y+H0WSR4CIsgThKKmp120pLIjRicbgSBy"
        "EZkOLrcRUDvXwO/xsa7692EQsDb3fi+OLBjG8mLBsHbcrHgDGVGh8bFdSYubI0eagRRnMc1d0nRL2odVGr08sPWVA0XIooKT"
        "dMdojb5ujda0w+UdIa7IBsmsHbOrUtoU5Xf81e8u8DVTng9PHAWlZ0KvHGcfNAxwi38h+R5///znoo2gYdQBZP7RjY3A/42u"
        "v7i4GN+ODjM67yG87YBC0P62KjEqW6rLdiWmVRf8O79jalz6gfuWyWvK8OMx7ZXK88RbtyrYvJ1yKjJJ3USO+0w7IBKQ/iw1"
        "uDFN8W4v4jNgpPpNqOIh3rnkkz8h6/1Nz1SOq+aZi7IgKnNvjDBwgKjIK5S1/I1+x8FnNcGW8xTA7VYxl+okv+J0Kd7qlpGz"
        "j++xkXlfoDaI9B2FNx+gySON3vSVEM4e1M2w8VbBvnNUo3u+oklcN8xylam5kn57WKjEgnGm/v9leG2EKIWL4EB8vhfUBBXp"
        "AlCukDWkTOXPfWCzU8qckDVwAtVnK3x/MGUYVYqYa20bHYSJMUiFBQc2WuK3k3wngRI0upkmHmuHO60AxgkOEKP8rBxR1YUj"
        "cwr2EMCBRXiRCkXMput20PJ0ZEK35/pwHGm3826OSUUqT++y4SaMg8e71Rz1d7l2LT0Fky2Y0BPm2Q9v03zi9V/2ffIyRTcD"
        "28q+/M4bU7ui94Rdb5ZR9mtvYea/jhsWfG1pZrizKmMyWIe1j+bTplP2eCzYxlrG6SIx7WqrTx1w8/W7VHEyGUEHtleGbJZz"
        "1IkHucBrINuDOh/gl7tNFiXr3vdtuUn10rigG+JZWpxG4r9NA8nSqAmH1fAn4Q5msR5oeUx5cMxQ0yTH9beyI/226lELiZkL"
        "gHLTrfA8H7h/A0DzdYhazyUe0F0BvLwSb99/FO0Xq98I0HRfI6qgABbCFntQmklm5K3Yfi5kcruorQHdVxQ61Y9ybTVH5doK"
        "fN5X/EGmXiCdT2nAA6rKa/x7eIXfr0b2fYS2nU1p6XVT8KEHPoS8c8u8KRYlGXKhTmqH8eS1LQKvDr4Dgye3T0C2NzgiTN/I"
        "KGY1ADcFw4dPFwNe7y0GKB5gM6B4MwzE0Ui8VCwPTNMralKc5ykLlLUi4hn9XtipPzRTYe1ehYcNgdeWgOrplCGAGUzvvJuH"
        "KXBbQ3zdfFnQRcE7Oz36ywK5WIyAFtUgA/1BkPhhsveZtHVIL533CYqRkRj3XhIuLDPoJjbIwwb8BgkSwX/XwQAD/bGEevzM"
        "j71oUhO14UqQhsV1XaqVA35SG08X7nsK7gG/+Heo+j1/3/xpsdo/GZ37ZzIH9s9VDNpXC9JE9hWDdjwU9lirWIGH6mk5PRSu"
        "zOe0ipKZG2ijuRVd71PiKxIL5zm9viH9ql69JyIN/Mlq4Jbiqr3wLkjD4+lUcil1su4pCKdt48JlXET3JdnxkuCqEx5ftSSE"
        "42wrI9XPrp6rl3ykcrSrttIr43PsCKllXFvPtW5WKP12ZjBn3cFONqAyAa32Ee3NUxQ4otH3QMAfcV6lzQ5h1b1N2jwCIofH"
        "+/XJ3tA3Q1x/1Xvwn6oT9NpIxrmNI2srWUTtFujZUHHs+NEuvdGypdr0BjZ8varJbAIhTa+NU+gAyx7hGZP50xfUT1WBN9L2"
        "i0kH+tYCcycjozntd/SeVaS2u6NO+3R+qF4t7NMTrlZYSuyBWhrho+hFRTyiSuDnlP/8DQUfFPFaC0FY9zonu8BzDuqnFofI"
        "/FsXgEXbS45a9WW7yL/BGrfu/EEZDModmqIyQfL2j8yOnaqbwzwGkWmNZ/jjFKD33ml3s+1ydHPq7h9KKAWB8dGYpMuJpn5l"
        "WTIK9BMSdyl3qHqK4RY1uEX/SY9COwiWHu3gntog8YjQOo3aq7gGfsuVqGbpLoseLebADxsw0B+Kmtb2dt16QKQUYjKT6lXy"
        "Dla0YSvMYGyuzQMesun9C3fyZq0="
    ),
}
print("Đã nạp bản sao nhúng:", ", ".join(EMBEDDED_SOURCES))

In [ ]:
# @title Nạp fusion.metrics và vision.sku110k_subset
import base64
import importlib
import importlib.util
import sys
import zlib

FALLBACK_DIR = Path("/content/_embedded_fallback")


def load_with_fallback(module_name, embedded_key):
    # Ưu tiên module trong repo; nếu thiếu thì nạp bản sao nhúng. Trả về (module, nguồn).
    try:
        if not repo_ok:
            raise ImportError("repo không có sẵn")
        return importlib.import_module(module_name), "repo"
    except Exception as exc:  # noqa: BLE001 - mọi lỗi import đều dẫn tới dự phòng
        print(f"Dùng bản nhúng cho {module_name} (lý do: {exc})")
        FALLBACK_DIR.mkdir(parents=True, exist_ok=True)
        path = FALLBACK_DIR / f"{embedded_key}.py"
        path.write_bytes(zlib.decompress(base64.b64decode(EMBEDDED_SOURCES[embedded_key])))
        spec = importlib.util.spec_from_file_location(embedded_key, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[embedded_key] = module
        spec.loader.exec_module(module)
        return module, "embedded"


if repo_ok and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

metrics, metrics_source = load_with_fallback("fusion.metrics", "fusion_metrics_embedded")
subset, subset_source = load_with_fallback("vision.sku110k_subset", "sku110k_subset_embedded")
print(f"fusion.metrics: {metrics_source} | sku110k_subset: {subset_source}")

## 1. Tải SKU-110K bằng loader chính thức của Ultralytics

Dùng `SKU-110K.yaml` (đi kèm Ultralytics): loader tải `SKU110K_fixed.tar.gz` (~13,6 GB) từ S3 của Trax,
giải nén và chuyển nhãn sang định dạng YOLO (1 lớp `object`; train 8.219 / val 588 / test 2.936 ảnh).

- Link S3 đôi khi **chập chờn**. Cell dưới kiểm tra dung lượng đĩa và link trước, và nếu tải lỗi sẽ báo
  rõ nguyên nhân kèm cách thử lại (chạy lại đúng cell này; loader sẽ tải lại từ đầu).
- Sau khi tải xong, chỉ **400 + 100 ảnh** được dùng; phần còn lại không tham gia huấn luyện.

In [ ]:
# @title Tải SKU-110K (có kiểm tra và báo lỗi rõ)
import shutil
import urllib.request

from ultralytics import settings
from ultralytics.data.utils import check_det_dataset

DATASETS_DIR.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(DATASETS_DIR).free / 1e9
print(f"Dung lượng đĩa trống: {free_gb:.1f} GB (cần >= {MIN_FREE_GB} GB)")
if free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f"Đĩa chỉ còn {free_gb:.1f} GB. Hãy dùng Runtime -> Disconnect and delete runtime rồi chạy lại."
    )

try:  # kiểm tra nhanh link S3 (chỉ cảnh báo, không dừng)
    request = urllib.request.Request(SKU_TAR_URL, method="HEAD")
    with urllib.request.urlopen(request, timeout=30) as response:
        size_gb = int(response.headers.get("Content-Length", 0)) / 1e9
        print(f"Link S3 phản hồi HTTP {response.status}, kích thước ~{size_gb:.1f} GB")
except Exception as exc:  # noqa: BLE001
    print(f"Cảnh báo: chưa kiểm tra được link S3 ({exc}). Vẫn thử tải bằng loader Ultralytics...")

settings.update({"datasets_dir": str(DATASETS_DIR)})
try:
    data_info = check_det_dataset("SKU-110K.yaml")  # tải + giải nén + chuyển nhãn nếu chưa có
    SKU_ROOT = Path(data_info["path"])
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "Tải SKU-110K thất bại: " + repr(exc) + "\n"
        "Cách xử lý:\n"
        "  1) Link S3 của SKU-110K đôi khi chập chờn: đợi vài phút rồi chạy lại cell này.\n"
        "  2) Nếu lỗi lặp lại, xoá dữ liệu dở dang rồi chạy lại: shutil.rmtree(DATASETS_DIR, ignore_errors=True).\n"
        "  3) Lỗi thiếu thư viện (polars/pandas): chạy lại cell này một lần nữa.\n"
        "  4) Hết đĩa: Runtime -> Disconnect and delete runtime."
    ) from exc
print("SKU-110K nằm ở:", SKU_ROOT)

## 2. Kiểm tra dữ liệu và cắt tập con cố định

`build_subset` lấy ngẫu nhiên **có seed** `N_TRAIN` ảnh từ `train.txt` và `N_VAL` ảnh từ `val.txt`
(hai tập vốn tách biệt nên không bị rò rỉ), chỉ nhận ảnh có nhãn. Chạy lại với cùng `SEED` luôn cho cùng
một tập con; danh sách tên ảnh được lưu để tái lập.

In [ ]:
# @title Kiểm tra gói dữ liệu đầy đủ và cắt tập con
import numpy as np

full_counts = subset.validate_sku110k_root(SKU_ROOT, min_images=MIN_FULL_IMAGES)
print("Số ảnh trong gói đầy đủ:", full_counts)

chosen = subset.build_subset(SKU_ROOT, SUBSET_DIR, n_train=N_TRAIN, n_val=N_VAL, seed=SEED)
subset.verify_subset(SUBSET_DIR, N_TRAIN, N_VAL)
DATA_YAML = subset.write_data_yaml(SUBSET_DIR)

for split, images in chosen.items():
    (SUBSET_DIR / f"{split}_images.txt").write_text("\n".join(p.name for p in images) + "\n", encoding="utf-8")
    gt_per_image = np.array([subset.count_label_boxes(subset.label_path_for_image(p)) for p in images])
    print(
        f"{split:5s}: {len(images)} ảnh | vật thể/ảnh: trung bình {gt_per_image.mean():.1f}, "
        f"nhỏ nhất {gt_per_image.min()}, lớn nhất {gt_per_image.max()}"
    )
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
# @title Xem thử vài ảnh val kèm nhãn gốc
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

val_dir = SUBSET_DIR / "images" / "val"
val_image_paths = sorted(p for p in val_dir.iterdir() if p.suffix.lower() in subset.IMAGE_SUFFIXES)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, image_path in zip(axes, val_image_paths[:2]):
    image = Image.open(image_path).convert("RGB")
    gt = subset.read_yolo_labels(SUBSET_DIR / "labels" / "val" / f"{image_path.stem}.txt")
    draw = ImageDraw.Draw(image)
    for x1, y1, x2, y2 in subset.yolo_to_xyxy(gt, *image.size):
        draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=2)
    ax.imshow(image)
    ax.set_title(f"{image_path.name}: {len(gt)} vật thể (nhãn gốc)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Huấn luyện YOLOv8n

Fine-tune từ trọng số COCO `yolov8n.pt` với đúng cấu hình đã chốt (`epochs=10`, `imgsz=640`, `batch=16`,
`seed=0`). Không tinh chỉnh siêu tham số, không xuất ONNX. Kết quả (`best.pt`, `results.csv`, biểu đồ)
nằm trong `/content/runs/sku110k_yolov8n/`.

In [ ]:
# @title Train
from ultralytics import YOLO

model = YOLO(BASE_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    device=DEVICE,
    workers=WORKERS,
    max_det=MAX_DET,   # validation trong lúc train cũng cần max_det lớn
    cache=True,        # 500 ảnh nhỏ -> cache RAM cho nhanh
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

SAVE_DIR = Path(model.trainer.save_dir)
BEST_PT = SAVE_DIR / "weights" / "best.pt"
RESULTS_CSV = SAVE_DIR / "results.csv"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"Không thấy {BEST_PT}: huấn luyện chưa hoàn tất?")
print("Kết quả huấn luyện ở:", SAVE_DIR)

In [ ]:
# @title Đường cong huấn luyện (từ results.csv)
import matplotlib.pyplot as plt
import pandas as pd

history = pd.read_csv(RESULTS_CSV)
history.columns = history.columns.str.strip()
curve_groups = [
    ("Loss (box)", ["train/box_loss", "val/box_loss"]),
    ("mAP (val)", ["metrics/mAP50(B)", "metrics/mAP50-95(B)"]),
]
fig, axes = plt.subplots(1, len(curve_groups), figsize=(12, 4))
for ax, (title, columns) in zip(axes, curve_groups):
    for column in columns:
        if column in history:
            ax.plot(history["epoch"], history[column], marker="o", label=column)
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
display(history.tail(3))

## 4. Đánh giá (val) bằng `best.pt`

- **mAP của Ultralytics**: `model.val()` (với `conf=0.001`, `max_det=1000`) rồi đưa kết quả qua
  `fusion.metrics.map_wrapper(ultralytics_results=...)` để có định dạng chuẩn của dự án.
- mAP thấp sau 10 epoch trên 400 ảnh là **bình thường**; mục tiêu chỉ là chạy thông quy trình.

In [ ]:
# @title Val bằng Ultralytics + map_wrapper (S02-C1)
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=0.001,
    iou=NMS_IOU,
    max_det=MAX_DET,
    device=DEVICE,
    workers=WORKERS,
    plots=False,
    project=str(RUNS_DIR),
    name="val",
    exist_ok=True,
)
map_ultralytics = metrics.map_wrapper(ultralytics_results=val_metrics)
print(
    f"[Ultralytics -> map_wrapper] mAP@0.5 = {map_ultralytics['map50']:.4f} | "
    f"mAP@0.5:0.95 = {map_ultralytics['map50_95']:.4f} | "
    f"precision = {map_ultralytics['precision']:.4f} | recall = {map_ultralytics['recall']:.4f}"
)

## 5. Dự đoán và chỉ số đếm (Counting)

Với mỗi ảnh val: **số đếm thật** = số hộp trong file nhãn; **số đếm dự đoán** = số hộp có
`conf >= COUNT_CONF`. Mô hình chỉ dự đoán **một lần** ở `conf=0.001` (giữ đủ hộp cho mAP), rồi lọc theo
ngưỡng để đếm. Sau đó dùng `fusion.metrics`: `counting_mae`, `counting_rmse`, `exact_accuracy`,
`evaluate_counting` và `map_wrapper` (mAP tự tính bằng NumPy, đối chiếu với số của Ultralytics).

> Lưu ý khi đọc kết quả: ảnh có ~147 vật thể nên **Exact Accuracy** (đếm đúng tuyệt đối) gần như luôn rất
> thấp; hãy nhìn MAE cùng với số vật thể trung bình (cột `MAE / trung bình GT`).

In [ ]:
# @title Dự đoán trên tập val
gt_boxes, detections, image_names = [], [], []
for image_path in val_image_paths:
    result = best_model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=0.001,
        iou=NMS_IOU,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False,
    )[0]
    height, width = result.orig_shape
    detections.append(subset.detections_to_array(result.boxes.xyxy, result.boxes.conf, result.boxes.cls))
    gt = subset.read_yolo_labels(SUBSET_DIR / "labels" / "val" / f"{image_path.stem}.txt")
    gt_boxes.append(subset.yolo_to_xyxy(gt, width, height))
    image_names.append(image_path.name)

print(f"Đã dự đoán {len(detections)} ảnh | số hộp thô (conf>=0.001) trung bình/ảnh: "
      f"{np.mean([len(d) for d in detections]):.0f}")

In [ ]:
# @title Chỉ số đếm bằng fusion.metrics
import json

gt_counts = [len(boxes) for boxes in gt_boxes]
thresholds = sorted({float(t) for t in CONF_SWEEP} | {float(COUNT_CONF)})
counts_by_conf = subset.counts_for_thresholds(detections, thresholds)

rows = []
for thr in thresholds:
    stats = metrics.evaluate_counting(gt_counts, counts_by_conf[thr])
    rows.append(
        {
            "conf": thr,
            "MAE": stats["mae"],
            "RMSE": stats["rmse"],
            "MAE / trung bình GT": stats["mae"] / float(np.mean(gt_counts)),
            "Exact Accuracy": stats["exact_accuracy"],
            "undercount_rate": stats["undercount_rate"],
            "overcount_rate": stats["overcount_rate"],
        }
    )
counting_table = pd.DataFrame(rows).set_index("conf")
display(counting_table.round(4))

pred_counts = counts_by_conf[float(COUNT_CONF)]
counting_main = {
    "conf": float(COUNT_CONF),
    "mae": metrics.counting_mae(gt_counts, pred_counts),
    "rmse": metrics.counting_rmse(gt_counts, pred_counts),
    "exact_accuracy": metrics.exact_accuracy(gt_counts, pred_counts),
    "mean_gt_count": float(np.mean(gt_counts)),
    "mean_pred_count": float(np.mean(pred_counts)),
    "n_images": len(gt_counts),
}
print(
    f"[conf={COUNT_CONF}] Counting MAE = {counting_main['mae']:.2f} vật thể/ảnh | "
    f"RMSE = {counting_main['rmse']:.2f} | Exact Accuracy = {counting_main['exact_accuracy']:.2%} | "
    f"GT trung bình = {counting_main['mean_gt_count']:.1f}, dự đoán trung bình = {counting_main['mean_pred_count']:.1f}"
)

val_counts = pd.DataFrame({"image": image_names, "gt_count": gt_counts, "pred_count": pred_counts})
val_counts["error"] = val_counts["pred_count"] - val_counts["gt_count"]
val_counts_csv = SAVE_DIR / "val_counts.csv"
val_counts.to_csv(val_counts_csv, index=False)
display(val_counts.head())

In [ ]:
# @title mAP tự tính bằng map_wrapper (NumPy) và đối chiếu với Ultralytics
import time

started = time.time()
map_standalone = metrics.map_wrapper(predictions=detections, ground_truths=gt_boxes, box_format="xyxy")
print(f"map_wrapper (NumPy) chạy trong {time.time() - started:.0f}s")

comparison = pd.DataFrame(
    {
        "mAP@0.5": [map_ultralytics["map50"], map_standalone["map50"]],
        "mAP@0.5:0.95": [map_ultralytics["map50_95"], map_standalone["map50_95"]],
    },
    index=["Ultralytics model.val()", "map_wrapper (NumPy, conf>=0.001)"],
)
display(comparison.round(4))
print("Hai cách nội suy/ghép cặp hộp khác nhau nên số liệu gần nhau chứ không trùng khớp tuyệt đối.")

In [ ]:
# @title Trực quan hoá dự đoán (conf = COUNT_CONF) so với nhãn gốc
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, image_path, gt_count in zip(axes, val_image_paths[:3], gt_counts[:3]):
    shown = best_model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=COUNT_CONF,
        iou=NMS_IOU,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False,
    )[0]
    ax.imshow(shown.plot(labels=False, conf=False, line_width=1)[:, :, ::-1])
    ax.set_title(f"{image_path.name}\nGT = {gt_count} | dự đoán = {len(shown.boxes)}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Lưu tóm tắt và tải kết quả về máy

Cell đầu ghi `s02_c3_summary.json` (cấu hình + mAP + chỉ số đếm). Cell thứ hai tải về máy:
`best.pt`, `results.csv`, `s02_c3_summary.json`, `val_counts.csv`. Nếu trình duyệt chặn nhiều tệp tải
cùng lúc, hãy cho phép "tải nhiều tệp" cho trang Colab rồi chạy lại cell.

In [ ]:
# @title Ghi tóm tắt kết quả (JSON)
summary = {
    "task": "S02-C3",
    "config": {
        "seed": SEED, "n_train": N_TRAIN, "n_val": N_VAL, "epochs": EPOCHS, "imgsz": IMGSZ,
        "batch": BATCH, "max_det": MAX_DET, "nms_iou": NMS_IOU, "count_conf": COUNT_CONF,
        "base_weights": BASE_WEIGHTS,
    },
    "sources": {"fusion.metrics": metrics_source, "sku110k_subset": subset_source, "repo_commit": repo_commit},
    "map_ultralytics": {k: v for k, v in map_ultralytics.items() if not k.startswith("per_class")},
    "map_standalone": {k: v for k, v in map_standalone.items() if not k.startswith("per_class")},
    "counting": counting_main,
    "counting_by_conf": counting_table.reset_index().to_dict(orient="records"),
}
SUMMARY_JSON = SAVE_DIR / "s02_c3_summary.json"
SUMMARY_JSON.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
    encoding="utf-8",
)
print("Đã ghi:", SUMMARY_JSON)

In [ ]:
# @title Tải best.pt + results.csv về máy
try:
    from google.colab import files
except ImportError:
    print("Không chạy trên Colab: lấy tệp trực tiếp ở", SAVE_DIR)
else:
    for path in (BEST_PT, RESULTS_CSV, SUMMARY_JSON, val_counts_csv):
        files.download(str(path))

## Khắc phục sự cố thường gặp

| Hiện tượng | Cách xử lý |
|---|---|
| `RuntimeError: Chưa bật GPU` | `Runtime → Change runtime type → T4 GPU`, chạy lại từ đầu |
| Tải SKU-110K lỗi / treo (link S3 chập chờn) | Đợi vài phút, chạy lại cell tải; nếu lặp lại thì xoá `DATASETS_DIR` rồi chạy lại |
| `CUDA out of memory` | Giảm `BATCH` (8 hoặc 4) ở cell cấu hình |
| Số đếm dự đoán thấp bất thường | Kiểm tra `MAX_DET` (phải ~1000) và bảng chỉ số theo `conf` |
| Mất kết quả sau khi phiên kết thúc | Chạy cell tải về ngay sau khi train xong |
| `git clone` lỗi | Bình thường: notebook tự dùng bản sao nhúng (xem dòng `fusion.metrics: embedded`) |

## Bước tiếp theo (ngoài phạm vi S02-C3)
Tinh chỉnh siêu tham số, fine-tune trên dữ liệu kệ hàng tự gán nhãn (S02-C2) và xuất ONNX **không** nằm
trong notebook này.